# 01 - Data Exploration

**Goal:** Understand the structure of Duolingo's notification dataset before implementing anything.

This notebook answers:
- What does each row represent?
- What are the "arms" (templates) of the bandit?
- How do eligibility and history work?
- What does the reward look like?
- What's the difference between train and test sets?

## Setup

We add the project root to `sys.path` so we can import from `src/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample, parse_history, parse_eligible_templates

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("Ready!")

---
## 1. Load Training Data (100K sample)

Each row = **one notification event**: Duolingo sent one push notification to one user at one moment. The data was collected under a **random policy** (templates chosen uniformly at random from eligible ones). This is critical — it means the observed reward rates are unbiased.

In [ ]:
train_df = load_sample(n_rows=100000, split="train")
print(f"\nShape: {train_df.shape}")
train_df.head(10)

### Save a 10K sample for quick access

We save a small 10K-row parquet file under `data/` so other notebooks or scripts can load it instantly without reading the full dataset every time.

In [14]:
sample_10k = train_df.head(10_000)
sample_path = os.path.join("..", "data", "sample_10k.parquet")
sample_10k.to_parquet(sample_path, index=False)
print(f"Saved {len(sample_10k):,} rows to {sample_path}")
print(f"File size: {os.path.getsize(sample_path) / 1e6:.2f} MB")

Saved 10,000 rows to ../data/sample_10k.parquet
File size: 1.11 MB


---
## 2. Column-by-Column Breakdown

| Column | Type | Meaning |
|--------|------|---------|
| `datetime` | float | Days since dataset start (e.g., 3.75 = day 3, 6pm) |
| `ui_language` | string | User's app language (proxy for user segment) |
| `eligible_templates` | list of strings | Which templates **could** be sent right now ("sleeping arms") |
| `history` | list of (template, days_ago) | What templates this user received recently |
| `selected_template` | string | Which template was actually sent (A–L) |
| `session_end_completed` | 0 or 1 | **REWARD** — did user complete a lesson within 2 hours? |

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

---
## 3. The Arms: Notification Templates (A–L)

In bandit terminology, each **template is an arm**. The algorithm's job is to learn which arm to pull (which template to send) for each user at each moment.

In [ ]:
# How many unique templates (arms)?
templates = sorted(train_df["selected_template"].unique())
print(f"Number of arms (templates): {len(templates)}")
print(f"Template labels: {templates}")

# How often was each template selected? (should be ~uniform since random policy)
template_counts = train_df["selected_template"].value_counts().sort_index()
print(f"\nSelection counts per template:\n{template_counts}")

fig, ax = plt.subplots(figsize=(10, 4))
template_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Template Selection Frequency (Random Policy → should be ~uniform)")
ax.set_xlabel("Template")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## 4. Sleeping Arms: Eligible Templates

Not all 12 templates are available at every moment. Some templates "sleep" — they're temporarily ineligible (e.g., a template just sent yesterday can't be sent again today). This is why the column `eligible_templates` varies per row.

In [ ]:
# Parse eligible_templates and look at a few examples
print("First 5 rows — eligible_templates:")
for i in range(5):
    raw = train_df["eligible_templates"].iloc[i]
    parsed = parse_eligible_templates(raw)
    print(f"  Row {i}: {parsed}  ({len(parsed)} eligible out of 12)")

# Distribution of how many templates are eligible per event
n_eligible = train_df["eligible_templates"].apply(lambda x: len(parse_eligible_templates(x)))
print(f"\nEligible set size stats:")
print(n_eligible.describe())

fig, ax = plt.subplots(figsize=(10, 4))
n_eligible.value_counts().sort_index().plot(kind="bar", ax=ax, color="coral", edgecolor="white")
ax.set_title("How many templates are eligible per event?")
ax.set_xlabel("Number of eligible templates")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## 5. User History

Each row carries a `history` field: a list of `(template, days_ago)` tuples showing what notifications this user received recently. This is what the **recency penalty** layer uses — if template A was sent 0.5 days ago, it gets penalized to avoid notification fatigue.

In [ ]:
# Parse history and show examples
print("Examples of user history:")
shown = 0
for i in range(len(train_df)):
    h = parse_history(train_df["history"].iloc[i])
    if len(h) > 0 and shown < 5:
        print(f"\n  Row {i}: {len(h)} past notifications")
        for template, days_ago in h[:5]:  # show up to 5 entries
            print(f"    Template '{template}' was sent {days_ago:.2f} days ago")
        shown += 1
    if shown >= 5:
        break

# How many users have history vs no history?
hist_lengths = train_df["history"].apply(lambda x: len(parse_history(x)))
print(f"\n--- History length stats ---")
print(f"Users with NO history:    {(hist_lengths == 0).sum():,} ({(hist_lengths == 0).mean()*100:.1f}%)")
print(f"Users with some history:  {(hist_lengths > 0).sum():,} ({(hist_lengths > 0).mean()*100:.1f}%)")
print(f"\nHistory length distribution:")
print(hist_lengths.describe())

---
## 6. Reward: `session_end_completed`

The reward is binary: **1** if the user completed a lesson within 2 hours of receiving the notification, **0** otherwise. This is what the bandit algorithm optimizes — pick the template that maximizes this rate.

In [ ]:
# Overall reward rate
reward_rate = train_df["session_end_completed"].mean()
total_rewarded = train_df["session_end_completed"].sum()
print(f"Overall reward rate: {reward_rate:.4f} ({reward_rate*100:.2f}%)")
print(f"Rewarded events: {total_rewarded:,} out of {len(train_df):,}")

# Reward rate per template
reward_by_template = train_df.groupby("selected_template")["session_end_completed"].agg(["mean", "sum", "count"])
reward_by_template.columns = ["reward_rate", "n_rewarded", "n_total"]
reward_by_template = reward_by_template.sort_values("reward_rate", ascending=False)
print(f"\nReward rate per template (sorted best → worst):")
print(reward_by_template.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
reward_by_template["reward_rate"].sort_index().plot(kind="bar", ax=ax, color="seagreen", edgecolor="white")
ax.axhline(y=reward_rate, color="red", linestyle="--", label=f"Overall avg: {reward_rate:.4f}")
ax.set_title("Raw Reward Rate per Template")
ax.set_xlabel("Template")
ax.set_ylabel("Reward Rate")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Users & Languages

There is **no user ID column** — each row is treated independently. The `ui_language` field is the closest proxy for user segmentation.

In [ ]:
# Language distribution
lang_counts = train_df["ui_language"].value_counts()
print(f"Number of unique languages: {len(lang_counts)}")
print(f"\nTop 10 languages:")
print(lang_counts.head(10))

# Reward rate by language (top 10)
top_langs = lang_counts.head(10).index
reward_by_lang = train_df[train_df["ui_language"].isin(top_langs)].groupby("ui_language")["session_end_completed"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

lang_counts.head(10).plot(kind="bar", ax=axes[0], color="mediumpurple", edgecolor="white")
axes[0].set_title("Top 10 Languages by Event Count")
axes[0].set_ylabel("Count")

reward_by_lang.plot(kind="bar", ax=axes[1], color="orange", edgecolor="white")
axes[1].set_title("Reward Rate by Language (Top 10)")
axes[1].set_ylabel("Reward Rate")

plt.tight_layout()
plt.show()

---
## 8. Time Distribution

`datetime` is a float representing days since the start of the dataset. Let's see the time span and how events are distributed over time.

In [ ]:
print(f"Time range: day {train_df['datetime'].min():.2f} to day {train_df['datetime'].max():.2f}")
print(f"Span: {train_df['datetime'].max() - train_df['datetime'].min():.1f} days")

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(train_df["datetime"], bins=100, color="teal", edgecolor="white", alpha=0.8)
ax.set_title("Distribution of Events Over Time (Training Sample)")
ax.set_xlabel("Day")
ax.set_ylabel("Number of Events")
plt.tight_layout()
plt.show()

---
## 9. Test Data — Quick Comparison

The test set is used for **offline evaluation** via importance sampling. Let's load a sample and compare.

In [ ]:
test_df = load_sample(n_rows=100_000, split="test")
print(f"\nTest shape: {test_df.shape}")
test_df.head(5)

In [ ]:
# Side-by-side comparison
print("=" * 50)
print("TRAIN vs TEST comparison")
print("=" * 50)
print(f"{'Metric':<30} {'Train':>10} {'Test':>10}")
print("-" * 50)
print(f"{'Rows (sample)':<30} {len(train_df):>10,} {len(test_df):>10,}")
print(f"{'Full rows (approx)':<30} {'~87.7M':>10} {'~114.5M':>10}")
print(f"{'Time span (days)':<30} {train_df['datetime'].max()-train_df['datetime'].min():>10.1f} {test_df['datetime'].max()-test_df['datetime'].min():>10.1f}")
print(f"{'Reward rate':<30} {train_df['session_end_completed'].mean():>10.4f} {test_df['session_end_completed'].mean():>10.4f}")
print(f"{'Unique templates':<30} {train_df['selected_template'].nunique():>10} {test_df['selected_template'].nunique():>10}")
print(f"{'Unique languages':<30} {train_df['ui_language'].nunique():>10} {test_df['ui_language'].nunique():>10}")
print(f"{'Columns':<30} {train_df.shape[1]:>10} {test_df.shape[1]:>10}")

---
## 10. Summary

Here's what we learned about the data:

| Fact | Value |
|------|-------|
| **Each row** | One notification sent to one user at one time |
| **Arms (templates)** | 12, labeled A through L |
| **Sleeping arms** | Not all templates eligible at every event — varies per row |
| **History** | List of (template, days_ago) tuples per event |
| **Reward** | Binary — did user complete a lesson within 2 hours? |
| **User ID** | None — rows are independent, no cross-row user tracking |
| **Collection policy** | Random (uniform over eligible templates) |
| **Train set** | ~87.7M rows, ~15 days |
| **Test set** | ~114.5M rows, ~19 days |

**Key insight:** Because the data was collected under a random policy, we can use **importance sampling** to estimate what any other policy would have achieved — without running a new A/B test.

**Next notebook:** `02_baseline_and_reward_rates.ipynb` — compute the random policy baseline and raw template reward rates.